In [1]:
#formats the competitionData into tfRecords for RNN training, including blockwise feature normalization
baseDir = '/mnt/c/Users/tomeu/Desktop/Master/TFM'

In [2]:
import os
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

In [3]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

2025-10-23 08:11:47.606263: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:925] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-10-23 08:11:47.806840: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:925] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-10-23 08:11:47.806920: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:925] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [4]:
#from tensorflow.keras import mixed_precision
#mixed_precision.set_global_policy('mixed_float16')

In [5]:
import os
import tensorflow as tf
import scipy.io
import scipy.sparse as sp
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from g2p_en import G2p
import re
from neuralDecoder.datasets.speechDataset import PHONE_DEF, VOWEL_DEF, CONSONANT_DEF, SIL_DEF, PHONE_DEF_SIL
from itertools import product
import pandas as pd
from spektral.data import Dataset, Graph
from spektral.layers import GATConv
from tensorflow.keras import layers, models
from getSpeechSessionBlocks import getSpeechSessionBlocks
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

**Esto corresponde a una sola frase que ,en este caso,  son 438 time steps: unos 8.8s**

In [7]:
input_features[0].shape[0]

NameError: name 'input_features' is not defined

**Ejemplo de time_steps para train data file dia 26-05-22**

In [27]:
time_steps_por_dia=[]
for features in input_features:
        time_steps_por_dia.append(features.shape[0])
sum(time_steps_por_dia)

240166

**Train feature files generation**

In [5]:
from block_normalization import feature_block_normalization
def saveFeaturesperDay(sessionName, dataPath, featuresPath):
    dat = scipy.io.loadmat(sessionPath)
    norm_feats = feature_block_normalization(dat)
    
    t_series_per_channel = [[] for ch in range(128)]
    
    for trial in range(0,len(norm_feats)):
        for feature in range(128):
            
            tx1 = norm_feats[trial][:, feature]
            SBP = norm_feats[trial][:, feature + 128]
            x = np.stack([tx1, SBP], axis=-1)
            
            # Acumular en la lista del canal
            t_series_per_channel[feature].append(x)
            os.makedirs( featuresPath , exist_ok=True)

    t_series_per_channel = [np.concatenate(segments, axis=0) for segments in t_series_per_channel]
    fname = os.path.join(featuresPath,"features")
    np.save(fname,t_series_per_channel,allow_pickle=True)
    print("Features from session: "+sessionName+" saved!")

**Estructura matriz de adyacencia**

In [6]:
from adjacency_matrix import adjacency_matrix 
A = adjacency_matrix()

In [7]:
print(A)

  (0, 1)	1.0
  (0, 2)	1.0
  (0, 7)	1.0
  (0, 14)	1.0
  (0, 16)	1.0
  (1, 0)	1.0
  (1, 14)	1.0
  (1, 16)	1.0
  (2, 0)	1.0
  (2, 3)	1.0
  (2, 7)	1.0
  (2, 10)	1.0
  (2, 14)	1.0
  (3, 2)	1.0
  (3, 6)	1.0
  (3, 7)	1.0
  (3, 10)	1.0
  (3, 12)	1.0
  (4, 5)	1.0
  (4, 6)	1.0
  (4, 9)	1.0
  (4, 12)	1.0
  (4, 18)	1.0
  (5, 4)	1.0
  (5, 8)	1.0
  :	:
  (122, 124)	1.0
  (123, 121)	1.0
  (123, 122)	1.0
  (123, 124)	1.0
  (123, 125)	1.0
  (123, 126)	1.0
  (124, 109)	1.0
  (124, 110)	1.0
  (124, 112)	1.0
  (124, 121)	1.0
  (124, 122)	1.0
  (124, 123)	1.0
  (124, 125)	1.0
  (124, 126)	1.0
  (125, 123)	1.0
  (125, 124)	1.0
  (125, 126)	1.0
  (126, 110)	1.0
  (126, 112)	1.0
  (126, 123)	1.0
  (126, 124)	1.0
  (126, 125)	1.0
  (127, 111)	1.0
  (127, 113)	1.0
  (127, 114)	1.0


**SBP concatenation**

In [59]:
from block_normalization import feature_block_normalization
blockLists = getSpeechSessionBlocks()
t_series_per_channel = [[] for ch in range(128)]
for sessIdx in range(len(blockLists)):
    sessionName = blockLists[sessIdx][0]
    dataPath = baseDir + '/competitionData' #Raw data
    sessionPath = dataPath + '/' + 'train' + '/' + sessionName + '.mat'
    dat = scipy.io.loadmat(sessionPath)
    session_data =  feature_block_normalization(dat)

    for trial in range(0,len(session_data)):
        for feature in range(128):
            SBP = session_data['inputFeatures'][trial][:, feature + 128 ]
            t_series_per_channel[feature].extend(SBP)

**Pearson correlation**

In [8]:
A_weighted=np.load("A_weighted.npy")

In [60]:
A_corr = np.corrcoef(t_series_per_channel)

In [61]:
from scipy.sparse import coo_matrix
A = A.tocoo()
rows, cols = A.row, A.col
data = A_corr[rows, cols]
A_weighted = sp.coo_matrix((data, (rows, cols)), shape=A.shape)

In [ ]:
print(A_weighted)

In [9]:
adj_matrix_tensor = tf.constant(A_weighted)

2025-10-23 08:12:05.031848: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-23 08:12:05.034936: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:925] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-10-23 08:12:05.035020: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:925] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-10-23 08:12:05.035061: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:925] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been bui

In [10]:
PHONE_DEF_SIL = [
    'AA', 'AE', 'AH', 'AO', 'AW',
    'AY', 'B',  'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G',
    'HH', 'IH', 'IY', 'JH', 'K',
    'L', 'M', 'N', 'NG', 'OW',
    'OY', 'P', 'R', 'S', 'SH',
    'T', 'TH', 'UH', 'UW', 'V',
    'W', 'Y', 'Z', 'ZH', 'SIL'
]
phone_to_id = {phone: i for i, phone in enumerate(PHONE_DEF_SIL)}

def _convert_to_ascii(text):
        return [ord(char) for char in text]
    
def multi_session_data_generator(master_index, max_seq_len):
    """
    Generador simplificado: YA NO produce la matriz de adyacencia.
    """
    g2p = G2p()
    session_cache = {}
    for session_path,trial_idx in master_index:
        if session_path not in session_cache:
            print(f"Cargando y procesando nueva sesión: {Path(session_path).name}")
            dat = scipy.io.loadmat(session_path)
            # Aplicamos la normalización una sola vez por sesión
            session_cache[session_path] = feature_block_normalization(dat)
        session_data = session_cache[session_path]
        trial_idx = int(trial_idx)
        tx1 = session_data['inputFeatures'][trial_idx][:, :128]     
        SBP = session_data['inputFeatures'][trial_idx][:, 128:]
        input_feats = np.stack([tx1, SBP], axis=-1)
        transcription = session_data['transcriptions'][trial_idx]
        input_len = np.array([session_data['frameLens'][trial_idx]], dtype=np.int32)

        transcription = re.sub(r'[^a-zA-Z\- \']', '', transcription).replace('--', '').lower()
        
        phonemes = []
        if len(transcription) == 0:
            phonemes = ['SIL']
        else:
            raw_phonemes = g2p(transcription)
            for p in raw_phonemes:
                if p == ' ': phonemes.append('SIL')
                p = re.sub(r'[0-9]', '', p)
                if re.match(r'[A-Z]+', p): phonemes.append(p)
            phonemes.append('SIL')

        label_ids = np.array([phone_to_id[p] for p in phonemes], dtype=np.int32)
        label_len = np.array([len(label_ids)], dtype=np.int32)
        inputFeats = session_data['inputFeatures'][trial_idx]
        ceMask = np.zeros([inputFeats.shape[0]]).astype(np.float32)
        ceMask[0:session_data['frameLens'][trial_idx]] = 1

        paddedTranscription = np.zeros([maxSeqLen]).astype(np.int32)
        paddedTranscription[0:len(transcription)] = np.array(_convert_to_ascii(transcription))

        # El diccionario de salida es ahora más pequeño
        inputs = {
            "features_input": input_feats,
            "labels_input": label_ids,
            "input_length": input_len,
            "label_length": label_len,
            "transcription": paddedTranscription
        }
        dummy_output = np.zeros((1,), dtype=np.float32)

        yield inputs, dummy_output

In [11]:
maxSeqLen = 500
train_dataset = tf.data.Dataset.from_generator(
    lambda: multi_session_data_generator(train_indices,maxSeqLen),
    output_signature=(
        {
            "features_input": tf.TensorSpec(shape=(None, None, None), dtype=tf.float32),
            "labels_input": tf.TensorSpec(shape=(None,), dtype=tf.int32),
            "input_length": tf.TensorSpec(shape=(1,), dtype=tf.int32),
            "label_length": tf.TensorSpec(shape=(1,), dtype=tf.int32),
            "transcription": tf.TensorSpec(shape=(maxSeqLen,), dtype=tf.int32)
            
        },
        tf.TensorSpec(shape=(None,), dtype=tf.float32)
    )
)

val_dataset = tf.data.Dataset.from_generator(
    lambda: multi_session_data_generator(val_indices,maxSeqLen),
    output_signature=(
        {
            "features_input": tf.TensorSpec(shape=(None, None, None), dtype=tf.float32),
            "labels_input": tf.TensorSpec(shape=(None,), dtype=tf.int32),
            "input_length": tf.TensorSpec(shape=(1,), dtype=tf.int32),
            "label_length": tf.TensorSpec(shape=(1,), dtype=tf.int32),
            "transcription": tf.TensorSpec(shape=(maxSeqLen,), dtype=tf.int32)
        },
        tf.TensorSpec(shape=(None,), dtype=tf.float32)
    )
)

test_dataset = tf.data.Dataset.from_generator(
    lambda: multi_session_data_generator(val_indices,maxSeqLen),
    output_signature=(
        {
            "features_input": tf.TensorSpec(shape=(None, None, None), dtype=tf.float32),
            "labels_input": tf.TensorSpec(shape=(None,), dtype=tf.int32),
            "input_length": tf.TensorSpec(shape=(1,), dtype=tf.int32),
            "label_length": tf.TensorSpec(shape=(1,), dtype=tf.int32),
            "transcription": tf.TensorSpec(shape=(maxSeqLen,), dtype=tf.int32)
        },
        tf.TensorSpec(shape=(None,), dtype=tf.float32)
    )
)
def add_adj_matrix(inputs, outputs):
    inputs["adj_matrix_input"] = adj_matrix_tensor
    return inputs, outputs


In [12]:
master_index = []
from block_normalization import feature_block_normalization
blockLists = getSpeechSessionBlocks()
for sessIdx in range(len(blockLists)):
    sessionName = blockLists[sessIdx][0]
    dataPath = baseDir + '/competitionData' #Raw data
    sessionPath = dataPath + '/' + 'train' + '/' + sessionName + '.mat'
    dat = scipy.io.loadmat(sessionPath)
    session_data =  feature_block_normalization(dat)
    #trial_indices = list(range(0,len(session_data['inputFeatures'])))
    numTrials = len(session_data['inputFeatures'])
    for trial_idx in range(numTrials):
            master_index.append((str(sessionPath), trial_idx))
    print(f"Índice maestro creado con {len(master_index)} muestras totales.")
    #np.random.shuffle(trial_indices)
master_index = np.array(master_index)
subset_size = int(len(master_index) * 0.3)
master_index = master_index[:subset_size]

Índice maestro creado con 280 muestras totales.
Índice maestro creado con 640 muestras totales.
Índice maestro creado con 1060 muestras totales.
Índice maestro creado con 1240 muestras totales.
Índice maestro creado con 1600 muestras totales.
Índice maestro creado con 1960 muestras totales.
Índice maestro creado con 2360 muestras totales.
Índice maestro creado con 2720 muestras totales.
Índice maestro creado con 3040 muestras totales.
Índice maestro creado con 3360 muestras totales.
Índice maestro creado con 3680 muestras totales.
Índice maestro creado con 4160 muestras totales.
Índice maestro creado con 4520 muestras totales.
Índice maestro creado con 4880 muestras totales.
Índice maestro creado con 5280 muestras totales.
Índice maestro creado con 5680 muestras totales.
Índice maestro creado con 6080 muestras totales.
Índice maestro creado con 6280 muestras totales.
Índice maestro creado con 6680 muestras totales.
Índice maestro creado con 7000 muestras totales.
Índice maestro creado 

In [13]:
train_split = 0.7
validation_split = 0.15
split_point_1 = int(len(master_index) * train_split)
split_point_2 = int(len(master_index) * (train_split + validation_split))
train_indices = master_index[:split_point_1]   
val_indices = master_index[split_point_1:split_point_2] 
test_indices = master_index[split_point_2:]  

train_dataset = train_dataset.map(add_adj_matrix, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.cache()
train_dataset = train_dataset.repeat()
train_dataset = train_dataset.shuffle(buffer_size=len(train_indices))

val_dataset = val_dataset.map(add_adj_matrix ,num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.cache()
#val_dataset = val_dataset.repeat()

test_dataset = test_dataset.map(add_adj_matrix ,num_parallel_calls=tf.data.AUTOTUNE)
test_dataset = test_dataset.cache()

In [14]:
BATCH_SIZE = 4
train_dataset = train_dataset.padded_batch(
    batch_size=BATCH_SIZE,
    padded_shapes=(
        {
            "features_input": tf.TensorShape([None, 128, 2]), # El tiempo es variable
            "adj_matrix_input": tf.TensorShape([128, 128]), # Fija
            "labels_input": tf.TensorShape([None]), # La longitud de la etiqueta es variable
            "input_length": tf.TensorShape([1]), # Fija
            "label_length": tf.TensorShape([1]), # Fija
            "transcription": tf.TensorShape([maxSeqLen,])
        },
        tf.TensorShape([None,]) # Dummy output
    )
)
val_dataset = val_dataset.padded_batch(
    batch_size=BATCH_SIZE,
    padded_shapes=(
        {
            "features_input": tf.TensorShape([None, 128, 2]), # El tiempo es variable
            "adj_matrix_input": tf.TensorShape([128, 128]), # Fija
            "labels_input": tf.TensorShape([None]), # La longitud de la etiqueta es variable
            "input_length": tf.TensorShape([1]), # Fija
            "label_length": tf.TensorShape([1]), # Fija
            "transcription": tf.TensorShape([maxSeqLen,])
        },
        tf.TensorShape([None,]) # Dummy output
    )
)


test_dataset = test_dataset.padded_batch(
    batch_size=BATCH_SIZE,
    padded_shapes=(
        {
            "features_input": tf.TensorShape([None, 128, 2]), # El tiempo es variable
            "adj_matrix_input": tf.TensorShape([128, 128]), # Fija
            "labels_input": tf.TensorShape([None]), # La longitud de la etiqueta es variable
            "input_length": tf.TensorShape([1]), # Fija
            "label_length": tf.TensorShape([1]), # Fija
            "transcription": tf.TensorShape([maxSeqLen,])
        },
        tf.TensorShape([None,]) # Dummy output
    )
)

val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)
print("\nForma de un lote del dataset:")
for inputs, _ in train_dataset.take(1):
    for name, tensor in inputs.items():
        print(f"- {name}: {tensor.shape}")


Forma de un lote del dataset:
Cargando y procesando nueva sesión: t12.2022.04.28.mat
Cargando y procesando nueva sesión: t12.2022.05.05.mat


2025-10-23 08:13:38.049221: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:380] Filling up shuffle buffer (this may take a while): 281 of 1847


Cargando y procesando nueva sesión: t12.2022.05.17.mat
Cargando y procesando nueva sesión: t12.2022.05.19.mat


2025-10-23 08:13:46.967438: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:380] Filling up shuffle buffer (this may take a while): 1061 of 1847


Cargando y procesando nueva sesión: t12.2022.05.24.mat
Cargando y procesando nueva sesión: t12.2022.05.26.mat
- features_input: (4, 439, 128, 2)
- labels_input: (4, 38)
- input_length: (4, 1)
- label_length: (4, 1)
- transcription: (4, 500)
- adj_matrix_input: (4, 128, 128)


2025-10-23 08:13:54.989625: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:405] Shuffle buffer filled.


**Creacion del modelo**

In [15]:
class GATWrapper(layers.Layer):
    """
    Esta capa envuelve a GATConv para que acepte una lista de dos tensores
    como una única entrada, haciéndola compatible con TimeDistributed.
    """
    def __init__(self, channels, attn_heads, **kwargs):
        super().__init__(**kwargs)
        # Instanciamos la capa GATConv dentro del envoltorio
        self.channels = channels
        self.attn_heads = attn_heads
        self.gat_conv = GATConv(
            channels=channels,
            attn_heads=attn_heads,
            concat_heads=True
        )

    def call(self, inputs):
        
        # 'inputs' será una lista: [features_en_un_timestep, adj_matrix_del_lote]
        # Pasamos estos argumentos directamente a la capa GATConv
        return self.gat_conv(inputs)

    def compute_output_shape(self, input_shape):
        """
        Calcula la forma de la salida basándose en la forma de la entrada.
        TimeDistributed llamará a este método con la forma de un único 'slice' de tiempo.
        """
        # input_shape es una lista de dos formas: [features_shape, adj_shape]
        features_shape = input_shape[0]
        
        # La forma de entrada de las features será (batch_size, num_nodes, num_input_features)
        # La extraemos del 'slice' de tiempo que nos pasa TimeDistributed.
        batch_size = features_shape[0]
        num_nodes = features_shape[1]
        
        # La GATConv transforma la última dimensión (features).
        # El número de features de salida es channels * attn_heads porque concat_heads=True
        output_features = self.channels * self.attn_heads
        
        # Devolvemos la forma de salida final para un 'slice' de tiempo
        return (batch_size, num_nodes, output_features)
    def get_config(self):
        # 1. Llama al get_config de la clase padre para obtener la config base.
        config = super().get_config()
        # 2. Actualiza el diccionario con los argumentos del constructor de ESTA capa.
        config.update({
            "channels": self.channels,
            "attn_heads": self.attn_heads,
        })
        return config

In [16]:
num_classes = 40
n_classes_ctc = num_classes +1
def build_model(n_nodes, n_feat, hidden_attn_units, attn_heads,gru_units):
    # Definimos dos entradas para el modelo
    features_input = layers.Input(shape=(None, n_nodes, n_feat), name="features_input")
    adj_input = layers.Input(shape=(n_nodes, n_nodes), sparse=False, name="adj_matrix_input")
    labels_input = layers.Input(shape=(None,), dtype="int32", name="labels_input")
    input_length = layers.Input(shape=(1,), dtype="int32", name="input_length")
    label_length = layers.Input(shape=(1,), dtype="int32", name="label_length")

 
    time_steps = tf.shape(features_input)[1]
    adj_expanded = layers.Lambda(
        lambda x: tf.expand_dims(x, axis=1)
    )(adj_input)
    def tile_time_dimension(inputs):
        tensor_to_tile, t_steps = inputs
        # multiples=[1, T, 1, 1] le dice a tf.tile que repita
        # 1 vez en el lote, T veces en el tiempo, 1 vez en N, 1 vez en N
        return tf.tile(tensor_to_tile, multiples=[1, t_steps, 1, 1])
    adj_input_repeated = layers.Lambda(tile_time_dimension)(
        [adj_expanded, time_steps]
    )
    # Se usa la capa personalizada
    gat_wrapper = GATWrapper(channels=hidden_attn_units, attn_heads=attn_heads)

    gat_output = layers.TimeDistributed(gat_wrapper)([features_input, adj_input_repeated])
    
    pooled_output = tf.reduce_mean(gat_output, axis=2)
    temporal = layers.GRU(gru_units, return_sequences=True)(pooled_output)
    temporal = layers.Dropout(0.3, name='dropout')(temporal)


    logits = layers.Dense(n_classes_ctc, name="logits")(temporal)
    logits_for_ctc = tf.transpose(logits, perm=[1, 0, 2])

    loss = tf.nn.ctc_loss(
        labels=labels_input,
        logits=logits_for_ctc,
        label_length=tf.squeeze(label_length, axis=-1), # Quitar la última dimensión
        logit_length=tf.squeeze(input_length, axis=-1), # Quitar la última dimensión
        blank_index=num_classes # El índice del token 'blank'
    )
    loss = tf.reduce_mean(loss)
# -----------------------------
# Compilar modelo
# -----------------------------
    training_model = models.Model(
        inputs=[features_input, adj_input, labels_input, input_length, label_length],
        outputs=logits
    )
    training_model.add_loss(loss)
    training_model.compile(optimizer='adam')

    prediction_model = models.Model(
        inputs=[features_input, adj_input],
        outputs=logits
    )
    return training_model, prediction_model



In [17]:
H = 64
n_feat = 2
n_nodes = 128 
# --- Creación e inspección del modelo ---
tr_model,pred_model = build_model(n_nodes=n_nodes, n_feat=n_feat, hidden_attn_units=H, attn_heads=3,gru_units=256)

# Imprimimos el resumen para verificar las formas de salida
tr_model.summary()

Instructions for updating:
Prefer tf.tensor_scatter_nd_add, which offers the same functionality with well-defined read-write semantics.
Instructions for updating:
Prefer tf.tensor_scatter_nd_update, which offers the same functionality with well-defined read-write semantics.
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 features_input (InputLayer)    [(None, None, 128,   0           []                               
                                2)]                                                               
                                                                                                  
 adj_matrix_input (InputLayer)  [(None, 128, 128)]   0           []                               
                                                                                                  
 tf.compat.v1.sha

**Training**

In [18]:
checkpoint_cb = ModelCheckpoint(
    filepath="mejor_modelo_gat-{epoch:02d}-{val_loss:.4f}.keras",
    monitor='val_loss', 
    mode='min',
    save_best_only=True,
    verbose=1
)
early_stopping = EarlyStopping(monitor='val_loss',mode='min',patience=12, restore_best_weights=True)
reduce_lr_cb = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,  
    patience=5,  
    mode='min',
    verbose=1,
    min_lr=1e-7   
)

In [19]:
model = tr_model.fit(
    train_dataset,
    epochs=100,
    steps_per_epoch=len(train_indices), validation_data=val_dataset,callbacks = [checkpoint_cb,early_stopping ,reduce_lr_cb]
)

Epoch 1/100


/home/tomeu/miniconda3/envs/speechBCIGNN/lib/python3.9/site-packages/keras/engine/functional.py:559: UserWarning: Input dict contained keys ['transcription'] which did not match any model input. They will be ignored by the model.
  inputs = self._flatten_to_reference_inputs(inputs)
2025-10-23 08:13:58.787995: I tensorflow/stream_executor/cuda/cuda_dnn.cc:366] Loaded cuDNN version 8907
2025-10-23 08:13:58.871448: I tensorflow/stream_executor/cuda/cuda_blas.cc:1774] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


Cargando y procesando nueva sesión: t12.2022.05.26.mat loss: 118.8053      
Cargando y procesando nueva sesión: t12.2022.06.02.mat

Epoch 00001: val_loss improved from inf to 84.01137, saving model to mejor_modelo_gat-01-84.0114.keras
1847/1847 [==============================] - 4472s 2s/step - loss: 118.8053 - val_loss: 84.0114 - lr: 0.0010
Epoch 2/100
1847/1847 [==============================] - ETA: 0s - loss: 108.9147     
Epoch 00002: val_loss did not improve from 84.01137
1847/1847 [==============================] - 4374s 2s/step - loss: 108.9147 - val_loss: 86.0758 - lr: 0.0010
Epoch 3/100
1847/1847 [==============================] - ETA: 0s - loss: 108.8726     
Epoch 00003: val_loss improved from 84.01137 to 83.64740, saving model to mejor_modelo_gat-03-83.6474.keras
1847/1847 [==============================] - 4395s 2s/step - loss: 108.8726 - val_loss: 83.6474 - lr: 0.0010
Epoch 4/100
1847/1847 [==============================] - ETA: 0s - loss: 109.1499     
Epoch 00004: val_

KeyboardInterrupt: 

In [27]:
custom_objects = {
    "GATWrapper": GATWrapper
    # Si tuvieras más capas u optimizadores personalizados, irían aquí
}

# ---------------------------------------------------------------------------
#CARGAR EL MODELO
# ---------------------------------------------------------------------------

# Ruta a tu archivo de modelo guardado
model_filepath = "mejor_modelo_gat-06-77.7250.keras" # Reemplaza con tu ruta

print(f"Cargando modelo desde: {model_filepath}")
model = tf.keras.models.load_model(model_filepath, custom_objects=custom_objects)

Cargando modelo desde: mejor_modelo_gat-06-77.7250.keras


In [34]:
all_logits = []
all_logit_lengths = []
all_true_seqs = []
all_true_seq_lengths = []
all_transcriptions = []
for inputs_batch, _ in train_dataset:
    # `batch` es un diccionario con 'features_input', 'adj_matrix_input', 'labels_input', etc.
    inputs_for_prediction = {
        "features_input": inputs_batch["features_input"],
        "adj_matrix_input": inputs_batch["adj_matrix_input"]
    }
    batch_logits = pred_model.predict(inputs_for_prediction)
    all_logits.append(batch_logits)
    all_logit_lengths.append(inputs_batch["input_length"].numpy())
    all_true_seqs.append(inputs_batch["labels_input"].numpy())
    all_true_seq_lengths.append(inputs_batch["label_length"].numpy())
    all_transcriptions.append(inputs_batch["transcription"].numpy())

In [35]:
logits_list = [item for batch in all_logits for item in batch]
logit_lengths_list = [item for batch in all_logit_lengths for item in batch]
true_seqs_list = [item for batch in all_true_seqs for item in batch]
true_seq_lengths_list = [item for batch in all_true_seq_lengths for item in batch]

# b. Aplicar padding a los logits (ya que tu dataset puede no tener padding global)
max_logit_len = max([l.shape[0] for l in logits_list])
padded_logits = np.array([np.pad(l, [[0, max_logit_len - l.shape[0]], [0, 0]]) for l in logits_list])

# c. Concatenar todo en arrays de NumPy finales
final_logits = padded_logits
final_logit_lengths = np.concatenate(logit_lengths_list, axis=0).flatten()
final_true_seqs = np.concatenate(true_seqs_list, axis=0)
final_true_seq_lengths = np.concatenate(true_seq_lengths_list, axis=0).flatten()
final_transcriptions = np.concatenate(all_transcriptions, axis=0)
mi_infOut = {
    'logits': final_logits,
    'logitLengths': final_logit_lengths,
    'transciptions': final_true_seqs,
    'trueSeqLengths': final_true_seq_lengths,
    'transcriptions': final_transcriptions
    # Puedes añadir más claves si son necesarias para tus métricas
}

**Language Model**

In [22]:
import neuralDecoder.utils.lmDecoderUtils as lmDecoderUtils
lmDir = baseDir+'/languageModel'
ngramDecoder = lmDecoderUtils.build_lm_decoder(
    lmDir,
    acoustic_scale=0.8, #1.2
    nbest=1,
    beam=18
)

I1023 20:12:04.726969 3183779 brain_speech_decoder.h:52] Reading fst /mnt/c/Users/tomeu/Desktop/Master/TFM/languageModel/TLG.fst
I1023 20:26:07.458563 3183779 brain_speech_decoder.h:81] Reading symbol table /mnt/c/Users/tomeu/Desktop/Master/TFM/languageModel/words.txt


In [36]:
decoder_out = lmDecoderUtils.cer_with_lm_decoder(
     ngramDecoder, 
     mi_infOut, 
     outputType='speech_sil', # Asegúrate de que los parámetros coincidan
     blankPenalty=np.log(2)
 )

  0%|          | 0/397 [00:00<?, ?it/s]

In [37]:
def _ascii_to_text(text):
        endIdx = np.argwhere(text==0)
        return ''.join([chr(char) for char in text[0:endIdx[0,0]]])
trueTranscriptions = [[],[]]
decodedTranscriptions = [[],[]]
for x in range(mi_infOut['transcriptions'].shape[0]):
        trueTranscriptions[0].append(_ascii_to_text(mi_infOut['transcriptions'][x,:]))  
        decodedTranscriptions[0] = decoder_out['decoded_transcripts']

In [37]:
from neuralDecoder.utils.lmDecoderUtils import _cer_and_wer as cer_and_wer
cer, wer = cer_and_wer(decodedTranscriptions[0], trueTranscriptions[0], outputType='speech_sil', returnCI=True)

#print word error rate
print(wer)

(0.9991525423728813, 0.9973844241978848, 1.0)


In [38]:
print(cer)

(0.859671709044094, 0.8520553508485947, 0.8675329747407553)
